# Omnichannel Support Ticket Tracker
**Tools:** Python (pandas), SQL (SQLite), Excel export, Power BI-ready data  
**Type:** Self-built project, **sample (synthetic) data**

**Goal:** SLA-based ticket tracking across email, chat and phone - flag SLA breaches and pending tickets, and produce a weekly summary of volume, resolution time and SLA adherence.

**Notebook flow**
1. Build the ticket log (channel, priority, status, resolution time, SLA due date)
2. Flag SLA breaches and overdue pending tickets
3. Query the log with SQL
4. Weekly summary + charts
5. Export to Excel / CSV for the Power BI dashboard

In [ ]:
import numpy as np, pandas as pd, sqlite3
import matplotlib.pyplot as plt
from pathlib import Path

np.random.seed(42)
DATA = Path("data"); DATA.mkdir(exist_ok=True)

REPORT_DATE = pd.Timestamp("2026-10-26 09:00")   # "today" for the report
START       = pd.Timestamp("2026-08-03")          # 12 weeks of tickets
SLA_TARGET  = 90                                  # target SLA adherence (%)

# SLA (hours to resolve) by priority
SLA_HOURS = {"P1-Critical": 4, "P2-High": 8, "P3-Medium": 24, "P4-Low": 48}

## 1. Build the ticket log

In [ ]:
N = 600
priority = np.random.choice(list(SLA_HOURS), N, p=[.08, .22, .45, .25])
channel  = np.random.choice(["Email", "Chat", "Phone"], N, p=[.42, .33, .25])
category = np.random.choice(
    ["Delivery delay", "Refund", "Billing error", "Account access", "Product fault", "General query"],
    N, p=[.24, .18, .16, .14, .12, .16])
created = START + pd.to_timedelta(np.random.randint(0, 12*7*24*60, N), unit="m")

# typical resolution time (hours) - chat is fastest, email slowest
base   = pd.Series(priority).map({"P1-Critical": 2.0, "P2-High": 4, "P3-Medium": 11, "P4-Low": 22}).values
chan_f = pd.Series(channel).map({"Chat": 0.55, "Phone": 0.75, "Email": 1.1}).values
escalated = np.random.rand(N) < 0.07
res_hours = base * chan_f * np.random.lognormal(0, 0.55, N) * np.where(escalated, 1.8, 1.0)
stuck = np.random.rand(N) < 0.03            # a few tickets get stuck waiting on a customer / another team
res_hours = np.where(stuck, res_hours * 60, res_hours)

df = pd.DataFrame({
    "ticket_id": [f"TKT-{10000+i}" for i in range(N)],
    "created_at": created, "channel": channel, "category": category,
    "priority": priority, "escalated": escalated})
df["sla_hours"]   = df["priority"].map(SLA_HOURS)
df["sla_due_at"]  = df["created_at"] + pd.to_timedelta(df["sla_hours"], unit="h")
resolved_at       = df["created_at"] + pd.to_timedelta(res_hours, unit="h")

# anything not yet resolved by the report date is still pending
is_resolved = resolved_at <= REPORT_DATE
df["resolved_at"]      = resolved_at.where(is_resolved)
df["status"]           = np.where(is_resolved, "Resolved", "Pending")
df["resolution_hours"] = (df["resolved_at"] - df["created_at"]).dt.total_seconds() / 3600

df = df.sort_values("created_at").reset_index(drop=True)
df.head()

## 2. Flag SLA breaches and pending tickets

In [ ]:
def sla_flag(r):
    if r["status"] == "Resolved":
        return "Met" if r["resolved_at"] <= r["sla_due_at"] else "Breached"
    return "Pending - Overdue" if REPORT_DATE > r["sla_due_at"] else "Pending - Within SLA"

df["sla_status"] = df.apply(sla_flag, axis=1)
df["hours_overdue"] = np.where(df["sla_status"] == "Pending - Overdue",
                               (REPORT_DATE - df["sla_due_at"]).dt.total_seconds() / 3600, np.nan).round(1)
df["week_start"] = (df["created_at"].dt.normalize() - pd.to_timedelta(df["created_at"].dt.weekday, unit="D"))

print(df["sla_status"].value_counts())

## 3. SQL analysis (SQLite)

In [ ]:
con = sqlite3.connect(":memory:")
sql_df = df.copy()
for c in ["created_at", "sla_due_at", "resolved_at", "week_start"]:
    sql_df[c] = sql_df[c].astype(str)
sql_df["escalated"] = sql_df["escalated"].astype(int)
sql_df.to_sql("tickets", con, index=False)

def q(sql): return pd.read_sql_query(sql, con)

# SLA adherence by channel (adherence = Met / closed tickets)
q('''
SELECT channel,
       COUNT(*)                                              AS total_tickets,
       SUM(sla_status = 'Met')                               AS met,
       SUM(sla_status = 'Breached')                          AS breached,
       ROUND(100.0 * SUM(sla_status = 'Met')
             / SUM(sla_status IN ('Met','Breached')), 1)     AS sla_adherence_pct,
       ROUND(AVG(resolution_hours), 1)                       AS avg_resolution_hrs
FROM tickets
GROUP BY channel
ORDER BY sla_adherence_pct DESC;
''')

In [ ]:
# Breaches by priority
q('''
SELECT priority,
       SUM(sla_status = 'Breached') AS breaches,
       COUNT(*)                     AS tickets,
       ROUND(100.0 * SUM(sla_status = 'Breached') / COUNT(*), 1) AS breach_rate_pct
FROM tickets
GROUP BY priority
ORDER BY priority;
''')

In [ ]:
# Pending tickets already past their SLA due date - the "chase list"
overdue = q('''
SELECT ticket_id, channel, category, priority, created_at, sla_due_at, hours_overdue
FROM tickets
WHERE sla_status = 'Pending - Overdue'
ORDER BY hours_overdue DESC
LIMIT 15;
''')
overdue

## 4. Weekly summary dashboard (volume, resolution time, SLA adherence)

In [ ]:
closed = df[df["status"] == "Resolved"]
weekly = (df.groupby("week_start").agg(tickets=("ticket_id", "count"),
                                        pending=("status", lambda s: (s == "Pending").sum()))
          .join(closed.groupby("week_start").agg(avg_resolution_hrs=("resolution_hours", "mean"),
                                                 met=("sla_status", lambda s: (s == "Met").sum()),
                                                 closed=("ticket_id", "count"))))
weekly["sla_adherence_pct"] = (100 * weekly["met"] / weekly["closed"]).round(1)
weekly["avg_resolution_hrs"] = weekly["avg_resolution_hrs"].round(1)
weekly = weekly.reset_index()
weekly

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
lbl = weekly["week_start"].dt.strftime("%d %b")

ax[0].bar(lbl, weekly["tickets"], color="#4C78A8"); ax[0].set_title("Weekly ticket volume")
ax[1].plot(lbl, weekly["avg_resolution_hrs"], marker="o", color="#F58518"); ax[1].set_title("Avg resolution time (hrs)")
ax[2].plot(lbl, weekly["sla_adherence_pct"], marker="o", color="#54A24B")
ax[2].axhline(SLA_TARGET, ls="--", color="grey", label=f"Target {SLA_TARGET}%")
ax[2].set_title("SLA adherence (%)"); ax[2].legend()
for a in ax: a.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.savefig(DATA / "weekly_dashboard.png", dpi=150); plt.show()

In [ ]:
# Channel x status view
pd.crosstab(df["channel"], df["sla_status"], margins=True)

## 5. Export for Excel and Power BI

In [ ]:
df.to_csv(DATA / "ticket_log.csv", index=False)
weekly.to_csv(DATA / "weekly_summary.csv", index=False)

try:
    with pd.ExcelWriter(DATA / "ticket_tracker.xlsx", engine="openpyxl") as xw:
        df.to_excel(xw, sheet_name="Ticket_Log", index=False)
        weekly.to_excel(xw, sheet_name="Weekly_Summary", index=False)
        overdue.to_excel(xw, sheet_name="Overdue_Pending", index=False)
    print("Saved data/ticket_tracker.xlsx")
except ModuleNotFoundError:
    print("Install openpyxl (pip install openpyxl) to create the Excel file; CSVs were saved.")

## Power BI setup
1. **Get Data -> Text/CSV** -> load `data/ticket_log.csv`.
2. Create a Date table and relate it to `created_at`.
3. Add these DAX measures:

```DAX
Total Tickets   = COUNTROWS(ticket_log)
Closed Tickets  = CALCULATE([Total Tickets], ticket_log[status] = "Resolved")
SLA Met         = CALCULATE([Total Tickets], ticket_log[sla_status] = "Met")
SLA Adherence % = DIVIDE([SLA Met], [Closed Tickets])
Avg Resolution Hrs = AVERAGE(ticket_log[resolution_hours])
Overdue Pending = CALCULATE([Total Tickets], ticket_log[sla_status] = "Pending - Overdue")
```
4. Visuals: weekly volume (column), avg resolution (line), SLA adherence gauge/line vs target, channel slicer, overdue table.